# 01 · 数据准备（Colab）—— v2：DNS Challenge 真实噪声

下载语料、建清单、合成固定测试集。跟 [v1](../v1_thchs30_musan/01_data_prep.ipynb) 的
区别只有一处：**噪声和房间冲激响应换成 DNS Challenge 的真实录制数据**，不再用
MUSAN/RIRS_NOISES。干净语音仍然是 THCHS-30（中文，带转写，供 CER 评测用）。

## 为什么要有这一版

v1 的固定测试集从一开始就是 **100% 程序合成噪声**（`rtse.data.synth.make_noise()`
生成的 keyboard/cafeteria/hum/white/pink/car/babble），从未用过真实录制噪声评测过。
合成噪声频谱结构规整、统计特性单一，指标可能比真实场景乐观。这一版把训练和评测都
换成/加入 DNS Challenge 的真实录制噪声（来自 AudioSet + Freesound），验证这个怀疑
到底有没有依据、影响多大。

## 执行前先看这里

1. **先跑「配置」cell**，确认 `CODE_ROOT`（v1/v2 共用的代码包目录，不用重新上传）
   和 `DRIVE_ROOT`（本版本专属，与 v1 物理隔离）都对
2. `rtse-colab.zip` 必须已上传到 `CODE_ROOT` 下（如果你跑过 v1，这一步已经做过了）
3. **第一次强烈建议先把 `QUICK_TEST = True`** 验证全流程（不碰 DNS 数据，走合成噪声/RIR）

## 产出

| 产物 | 位置（都在 `DRIVE_ROOT` 下，持久） | 体积 |
|---|---|---|
| `manifest.json` | 项目根 | 几百 KB |
| `archives/*.tar.bz2` | 项目根（hybrid 模式） | 约 20 GB |
| `testset/` + `testset.zip` | 项目根 | 比 v1 略大（多了真实噪声分层） |

> ⚠️ 测试集必须固定下来，训练集必须在线随机混音——原因跟 v1 一样，见那边的说明。
> v2 的固定测试集**同时包含合成噪声分层（跟 v1 完全同构，可直接比）和真实噪声分层
> （新增）**，这样能在同一份报告里回答"合成噪声测出来好看多少"这个问题。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与选择都集中在这一个 cell，别处不要再写死路径
#  这是 v2（DNS Challenge 真实噪声）。v1（THCHS-30+MUSAN）在
#  notebooks/v1_thchs30_musan/，两版数据/checkpoint 各自独立，不会互相覆盖。
# ═══════════════════════════════════════════════════════════════════════

# rtse-colab.zip 所在目录——**v1/v2 共用同一份代码包**，不需要重复上传。
# 如果你还没跑过 v1，这里跟下面 DRIVE_ROOT 的上一级目录填一样的就行。
CODE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# 本版本专属的数据/checkpoint/测试集根目录，嵌套在 CODE_ROOT 下面，
# 与 v1 直接写在 CODE_ROOT 下的 manifest.json/checkpoints/models/testset 互不冲突。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE/v2_dns_real_noise'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 原始语料怎么放？────────────────────────────────────────────────────
#
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每次会话解压到本地盘。
#               一次下载永久有效；训练读取是本地盘全速；
#               每个新会话只需几分钟解压。
#
#   'local'  —— 全部放本地盘，压缩包用完即删。
#               不占 Drive；代价是**每次新会话都要重下**。
#
#   'drive'  —— 全部放 Drive。THCHS-30 一万多个小文件在 FUSE 挂载上解压很慢，
#               DNS 噪声/IR 是几个大文件（不是海量小文件）不受这个问题影响，
#               但仍不推荐——训练时随机读取 Drive 比本地盘慢。
DATA_MODE = 'hybrid'

# ── 快速验证模式 ────────────────────────────────────────────────────────
# True  = 只下载 337 MB 的小语料（LibriSpeech dev-clean，英文），不碰 DNS 数据，
#         约 15 分钟验证「数据→训练→导出→回传」整条链路。
#         **只用来验证流程，不要用它的结果做最终指标**。
# False = 完整流程：THCHS-30 中文语音 + DNS Challenge 真实噪声/RIR
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都要靠它

assert DATA_MODE in ('hybrid', 'local', 'drive'), f'DATA_MODE 只能是 hybrid/local/drive'

CODE = CODE_ROOT
DRIVE = DRIVE_ROOT
WORK = WORK_ROOT

# 压缩包放哪 / 解压到哪 —— 三种模式的唯一区别就在这两行
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{DRIVE}/rawdata' if DATA_MODE == 'drive' else f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'    # local 模式解压后删包省空间，其余保留以便复用

# Drive 侧的产物目录（**这些永远在 Drive 上**，训练结果不能放临时盘）
CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每个 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(CODE), (
    f'Drive 上找不到 {CODE}\n'
    '检查两件事：① Drive 已挂载成功；② CODE_ROOT 与你实际存放 rtse-colab.zip 的目录一致。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局（v2 · DNS Challenge 真实噪声）')
print('─' * 74)
print(f'  代码包(v1/v2 共用)  {CODE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(小语料/英文)" if QUICK_TEST else "完整流程(THCHS-30 中文语音 + DNS 真实噪声/RIR)"}')
print()

!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 CODE_ROOT 指向的目录（v1/v2 共用同一份，不用重新上传）。
ZIP = f'{CODE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {CODE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(CODE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没有预装的几个包。
# 不用 `pip install -e .`：那会去解析 pyproject 里锁定的 torch CPU 索引，
# 把 Colab 自带的 GPU 版 torch 覆盖掉 —— 训练会瞬间慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。如果 Colab 上的 STFT 与本地哪怕差一点，
# 训练出来的模型拿回本地就会掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 下载语料

| 用途 | 数据集 | 体积 | 为什么选它 |
|---|---|---|---|
| 中文语音 | **THCHS-30**（openSLR SLR18） | ~6.4 GB | 跟 v1 一样，不换——DNS Challenge 没有中文语料 |
| 噪声 | **DNS Challenge 5** `noise_fullband`：AudioSet + Freesound 各 1 个分片 | ~9 GB（估） | 真实录制，覆盖面比 MUSAN noise 子集广 |
| 房间冲激响应 | **DNS Challenge 5** `impulse_responses` 分片 | ~5.9 GB | 真实测量 + 仿真混合 |
| 冲激型噪声 | 本项目合成（跟 v1 一样） | 0 | 针对性压力测试（`docs/FINDINGS.md` F-02），真实噪声库不保证覆盖 |

**为什么只选 2 个噪声分片、1 个 IR 分片**：DNS Challenge 5 完整语料（多语言干净语音 +
全部噪声/IR 分片）总计接近 1TB，本地/Colab 都没必要下载完整版——这是"有代表性的小规模
子集"，不是"能拿到的全部数据"。分片是 Azure 公开 blob，直接 wget，支持断点续传。

In [ ]:
# 语料清单：THCHS-30（openSLR）+ DNS Challenge 噪声/IR 分片（Azure 公开 blob）

if QUICK_TEST:
    SPEECH_DATASETS = [
        ('librispeech', 12, 'dev-clean.tar.gz', 0.34, 'LibriSpeech'),
    ]
    DNS_NOISE_SHARDS, DNS_IR_SHARDS = [], []   # 快速模式不碰 DNS，走合成噪声/RIR
else:
    SPEECH_DATASETS = [
        ('thchs30', 18, 'data_thchs30.tgz', 6.4, 'data_thchs30'),
    ]
    # 2 个真实噪声分片（AudioSet 来源的日常环境声 + Freesound 来源的标注音效），
    # 覆盖面已经比 MUSAN 的 noise 子集广很多；1 个房间冲激响应分片（真实测量+仿真混合）。
    # "有代表性的小规模子集"：这 3 个分片总计约 20GB，相对完整 DNS5 语料（噪声+IR 约 64GB，
    # 加上多语言干净语音后总计约 890GB）是一个刻意选小的子集，不追求覆盖全部分片。
    # 单个分片体积未逐一实测（本机磁盘装不下拿来验证），下方按官方文档给出的
    # "全部噪声分片共 ~39GB / 9 个分片"估算均摊值，供磁盘预算做粗略预检，
    # 实际大小以下载时 wget 显示的为准。
    DNS_NOISE_SHARDS = [
        ('dns_noise_audioset', 'noise_fullband/datasets_fullband.noise_fullband.audioset_000.tar.bz2', 4.3),
        ('dns_noise_freesound', 'noise_fullband/datasets_fullband.noise_fullband.freesound_000.tar.bz2', 4.3),
    ]
    DNS_IR_SHARDS = [
        ('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2', 5.9),
    ]

need_gb = (sum(d[3] for d in SPEECH_DATASETS)
          + sum(s[2] for s in DNS_NOISE_SHARDS) + sum(s[2] for s in DNS_IR_SHARDS))
free_arch = shutil.disk_usage(ARCHIVE_DIR).free / 1e9
free_data = shutil.disk_usage(DATA).free / 1e9
print(f'需要约 {need_gb:.1f} GB（DNS 分片大小为估算值，实际以下载时显示的为准）')
print(f'  压缩包卷 {ARCHIVE_DIR}  可用 {free_arch:.1f} GB')
print(f'  解压卷   {DATA}  可用 {free_data:.1f} GB')
assert free_arch > need_gb * 1.15, '压缩包卷空间不够'
assert free_data > need_gb * 1.3, (   # DNS 分片解压比压缩比不如 openSLR 语料，余量给大一点
    '解压卷空间不够。Colab Pro 本地盘通常有 200+ GB；若确实不够，'
    '先设 QUICK_TEST=True 跑小语料验证流程，或减少 DNS_NOISE_SHARDS 的分片数。'
)

In [ ]:
MIRRORS = [
    'https://www.openslr.org/resources',
    'https://openslr.elda.org/resources',          # 欧洲
    'https://openslr.magicdatatech.com/resources', # 中国
]

def fetch_openslr(name, slr, fname, expect_dir):
    """下载 → 解压 → 校验（openSLR 镜像，.tgz/.zip）。跟 v1 是同一份逻辑。"""
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    archive = f'{ARCHIVE_DIR}/{fname}'

    if os.path.exists(ex_mark) and os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[skip  ] {name} 已解压'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)，跳过下载')
    else:
        ok = False
        for base in MIRRORS:
            url = f'{base}/{slr}/{fname}'
            print(f'[get   ] {name} ← {url}')
            rc = os.system(f'wget -q --show-progress -c -T 30 -O {shq(archive)} {shq(url)}')
            if rc == 0 and os.path.exists(archive) and os.path.getsize(archive) > 1e6:
                ok = True
                break
            print('[retry ] 该镜像失败，换下一个')
        if not ok:
            print(f'[FAIL  ] {name} 三个镜像都下不下来。'
                  f'去 https://www.openslr.org/{slr}/ 确认文件名是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {DATA}')
    t = time.time()
    if fname.endswith('.zip'):
        rc = os.system(f'unzip -q -o {shq(archive)} -d {shq(DATA)}')
    else:
        rc = os.system(f'tar -xzf {shq(archive)} -C {shq(DATA)}')

    if not os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[FAIL  ] 解压后没有找到 {DATA}/{expect_dir}（rc={rc}）')
        print(f'         压缩包可能不完整，删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive)
        Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   解压耗时 {time.time()-t:.0f} 秒')
    return True

DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

def fetch_dns_blob(name, blob_path, expect_min_wavs=1):
    """下载 → 解压 DNS Challenge 的一个 .tar.bz2 分片。

    跟 fetch_openslr 用同一套"下载标记 + 解压标记"续传逻辑，但解压用 bz2
    （`tar -xjf`），而且**不假设解压后的内部目录名**——DNS Challenge 官方仓库
    没有在文档里给出每个分片解压后的确切子目录结构，本机磁盘装不下几 GB 的
    分片来提前验证（见 docs/ENVIRONMENT.md，C: 盘预算 <5GB），所以校验方式
    改成"解压后目录树里递归扫到的 wav 数量"，而不是断言一个具体子目录名——
    这样即使 DNS 内部打包结构和预期不同，只要文件确实解出来了就能识别成功，
    下游 scan() 本来就是递归扫描，不关心具体嵌套了几层。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    if os.path.exists(ex_mark):
        n = count_wavs()
        if n >= expect_min_wavs:
            print(f'[skip  ] {name} 已解压（{n} 个 wav）'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)，跳过下载')
    else:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(archive)} {shq(url)}')
        if rc != 0 or not os.path.exists(archive) or os.path.getsize(archive) < 1e6:
            print(f'[FAIL  ] {name} 下载失败。'
                  f'去 https://github.com/microsoft/DNS-Challenge 确认 blob 路径是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    rc = os.system(f'tar -xjf {shq(archive)} -C {shq(out_dir)}')
    n = count_wavs()
    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav 文件（rc={rc}），压缩包可能不完整')
        print(f'         删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive)
        Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒')
    return True

t0 = time.time()
results = {n: fetch_openslr(n, s, f, d) for n, s, f, _, d in SPEECH_DATASETS}
for n, blob, _gb in DNS_NOISE_SHARDS:
    results[n] = fetch_dns_blob(n, blob, expect_min_wavs=100)
for n, blob, _gb in DNS_IR_SHARDS:
    results[n] = fetch_dns_blob(n, blob, expect_min_wavs=50)
print(f'\n总耗时 {(time.time()-t0)/60:.1f} 分钟   结果: {results}')
assert all(results.values()), '有数据集没就绪，看上面的 FAIL 信息'

!df -h {shq(DATA)} | tail -1
!ls -1 {shq(DATA)}

## 2. 建立文件清单（按说话人 / 按噪声文件划分）

In [ ]:
import random

def scan(root, exts=('.wav', '.flac')):
    root = Path(root)
    if not root.exists(): return []
    return sorted(str(p) for p in root.rglob('*') if p.suffix.lower() in exts)

if QUICK_TEST:
    speech_all = scan(f'{DATA}/LibriSpeech')
    noise_all = []          # 快速模式不下 DNS，只用合成噪声
    rir_all = []             # 也不下 DNS IR，用合成 RIR
else:
    speech_all = scan(f'{DATA}/data_thchs30/data')
    # scan() 是递归扫描，不关心 DNS 分片解压后内部套了几层目录
    noise_all = scan(f'{DATA}/dns_noise_audioset') + scan(f'{DATA}/dns_noise_freesound')
    rir_all = scan(f'{DATA}/dns_ir')

print(f'语音 {len(speech_all):>6} 条')
print(f'噪声 {len(noise_all):>6} 条' + ('   (快速模式：仅用合成噪声)' if QUICK_TEST else '   (DNS 真实噪声)'))
print(f'RIR  {len(rir_all):>6} 条' + ('   (快速模式：仅用合成 RIR)' if QUICK_TEST else '   (DNS 真实 IR)'))
assert speech_all, '没扫到语音文件，检查上一步的解压结果'
if not QUICK_TEST:
    assert noise_all, 'DNS 噪声分片解压后没扫到 wav，检查上一步 fetch_dns_blob 的输出'
    assert rir_all, 'DNS IR 分片解压后没扫到 wav，检查上一步 fetch_dns_blob 的输出'

In [ ]:
# 按**说话人**划分语音，按**文件**划分噪声/RIR——跟 v1 用的是同一个随机种子
# （20260805 / 42），保证说话人切分、噪声/RIR 的 train/test 划分跟 v1 完全一致，
# 差异只来自"噪声/RIR 换了真实数据源"这一个变量，其余条件对齐，比较才有意义。
def speaker_of(p):
    stem = Path(p).stem
    return stem.split('_')[0] if '_' in stem else stem.split('-')[0]

spk = sorted({speaker_of(p) for p in speech_all})
random.Random(20260805).shuffle(spk)
n_test = max(2, len(spk) // 10)
n_val = max(2, len(spk) // 10)
spk_test = set(spk[:n_test])
spk_val = set(spk[n_test:n_test + n_val])
spk_train = set(spk[n_test + n_val:])
print(f'说话人 {len(spk)} 位 → train {len(spk_train)} / val {len(spk_val)} / test {len(spk_test)}')

split = {'train': [], 'val': [], 'test': []}
for p in speech_all:
    s = speaker_of(p)
    split['test' if s in spk_test else 'val' if s in spk_val else 'train'].append(p)

rnd = random.Random(42)
nz = noise_all[:]; rr = rir_all[:]
rnd.shuffle(nz); rnd.shuffle(rr)
nz_test, nz_train = nz[:len(nz)//5], nz[len(nz)//5:]
rir_test, rir_train = rr[:len(rr)//5], rr[len(rr)//5:]

for k, v in split.items():
    print(f'  语音 {k:>5}: {len(v):>6} 条')
print(f'  噪声 train/test: {len(nz_train)}/{len(nz_test)}')
print(f'  RIR  train/test: {len(rir_train)}/{len(rir_test)}')

manifest = {
    'version': 'v2_dns_real_noise',
    'quick_test': QUICK_TEST,
    'data_dir': DATA,
    'speech': split,
    'noise_train': nz_train, 'noise_test': nz_test,
    'rir_train': rir_train, 'rir_test': rir_test,
    'speaker_split': {'train': sorted(spk_train), 'val': sorted(spk_val), 'test': sorted(spk_test)},
}

## 3. 补充冲激型噪声

跟 v1 完全一样，**不因为换了真实噪声源就删掉**。DNS Challenge 的 AudioSet/Freesound
噪声虽然真实、覆盖面广，但不保证包含足够密度的键盘敲击这类冲激噪声样本；
`docs/FINDINGS.md` F-02 已证明这类噪声会让所有 MCRA 类 DSP 方法失效，
是神经网络最有说服力的立足点之一，训练集里必须稳定含有它，不能依赖"广撒网式"真实噪声库
恰好采样到。

In [ ]:
import numpy as np, soundfile as sf
from rtse.data.synth import make_noise

IMPULSIVE = f'{DATA}/synth_impulsive'
os.makedirs(IMPULSIVE, exist_ok=True)
rng = np.random.default_rng(7)

made = []
for kind in ['keyboard', 'cafeteria', 'hum', 'white', 'pink', 'car', 'babble']:
    for i in range(30):                       # 每类 30 条 × 10 秒
        y = make_noise(kind, 16000 * 10, rng)
        y = y / (np.max(np.abs(y)) + 1e-9) * 0.7
        fn = f'{IMPULSIVE}/{kind}_{i:03d}.wav'
        sf.write(fn, y, 16000, subtype='PCM_16')
        made.append(fn)

rnd.shuffle(made)
cut = len(made) // 5
manifest['noise_test'] += made[:cut]
manifest['noise_train'] += made[cut:]

Path(f'{DRIVE}/manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
print(f'合成 {len(made)} 条冲激噪声 → 训练 +{len(made)-cut} / 测试 +{cut}')
print(f'清单已写入 {DRIVE}/manifest.json')
print(f'  训练噪声总数 {len(manifest["noise_train"])}，测试噪声总数 {len(manifest["noise_test"])}')

## 4. 合成固定测试集

跟 v1 同构的**合成噪声主表**（SNR × 噪声类型，T60=0.3）+ **混响扫描**照抄不改，
保证能跟 v1 直接对比。新增一组**真实噪声分层**：同样扫 SNR，噪声取自上面划分出来的
`nz_test`（训练阶段从未见过的真实 DNS 噪声文件），T60 固定 0.3 —— 这一层是 v2 独有的，
用来回答"这个模型在真实噪声上到底表现如何"。

音频长度沿用 v1 修复后的做法：按每条语音的自然时长保留完整内容（[3s, 18s] 保护），
不做固定秒数截断——原因见 `docs/ISSUES.md` I-21，这里直接复用已验证过的做法，
不重新踩一遍那个坑。

In [ ]:
from rtse.audio.io import read_audio, write_audio
from rtse.data.synth import mix_at_snr, apply_rir, make_rir
from rtse.vad import build_vad
from tqdm.auto import tqdm

SNRS = [-5, 0, 5, 10, 15, 20]
NOISES = ['keyboard', 'cafeteria', 'babble', 'white', 'car', 'hum']  # 合成噪声，跟 v1 完全一致
T60S = [0.0, 0.3, 0.6, 0.9]
PER_CELL = 20
MIN_SEG_SEC, MAX_SEG_SEC = 3.0, 18.0  # 自然时长的下限/上限保护，见上方说明

def read_transcript(wav_path):
    """读 THCHS-30 的转写（.wav.trn）。快速模式的 LibriSpeech 没有，返回 None。"""
    trn = Path(str(wav_path) + '.trn')
    if not trn.exists():
        return None
    lines = trn.read_text(encoding='utf-8').strip().splitlines()
    if not lines:
        return None
    first = lines[0].strip()
    if first.endswith('.trn'):
        real = Path(wav_path).parent.parent / first
        if real.exists():
            lines = real.read_text(encoding='utf-8').strip().splitlines()
    return lines[0].replace(' ', '') if lines else None

test_speech = [p for p in split['test'] if read_transcript(p)] or split['test']
has_text = read_transcript(test_speech[0]) is not None
print(f'测试集可用语音 {len(test_speech)} 条，带转写: {has_text}')
if not has_text:
    print('⚠ 没有转写文本 → 后续算不了 CER。快速模式下属正常。')

cells = [{'snr': s, 'noise': n, 't60': 0.3, 'real_noise': False} for s in SNRS for n in NOISES]
cells += [{'snr': 5, 'noise': 'babble', 't60': t, 'real_noise': False} for t in T60S if t != 0.3]
# 真实噪声分层：只有非快速模式、且确实划出了测试用的真实噪声文件才加
if not QUICK_TEST and nz_test:
    cells += [{'snr': s, 'noise': 'dns_real', 't60': 0.3, 'real_noise': True} for s in SNRS]
n_samples = len(cells) * PER_CELL
print(f'{len(cells)} 格 × {PER_CELL} 条 = {n_samples} 个样本'
      f'（其中真实噪声分层 {len(SNRS) if (not QUICK_TEST and nz_test) else 0} 格）')

In [ ]:
def real_noise_clip(rng_, seg_len_):
    """从留出的真实 DNS 噪声测试子集(nz_test)里取一条，裁/拼到 seg_len_ 长度。"""
    y = np.zeros(0)
    for _ in range(min(20, len(nz_test))):
        p = nz_test[rng_.integers(len(nz_test))]
        y = read_audio(p)
        if y.size >= 16000 * 0.5:   # 太短(<0.5s)的片段跳过，可能是切碎的边角料
            break
    if y.size == 0:
        y = np.zeros(seg_len_)
    if y.size < seg_len_:
        y = np.tile(y, int(np.ceil(seg_len_ / y.size)))
    start = rng_.integers(0, y.size - seg_len_ + 1) if y.size > seg_len_ else 0
    return y[start:start + seg_len_]


os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)
rng = np.random.default_rng(20260805)
records, sidx = [], 0
skipped_too_long = 0

for ci, cell in enumerate(tqdm(cells, desc='合成测试集')):
    for k in range(PER_CELL):
        idx0 = ci * PER_CELL + k
        for attempt in range(len(test_speech)):
            sp = test_speech[(idx0 + attempt) % len(test_speech)]
            clean = read_audio(sp)
            if clean.size / 16000 <= MAX_SEG_SEC:
                break
        else:
            continue
        if clean.size / 16000 > MAX_SEG_SEC:
            skipped_too_long += 1

        seg_len = int(np.clip(clean.size / 16000, MIN_SEG_SEC, MAX_SEG_SEC) * 16000)
        if clean.size < seg_len:
            clean = np.pad(clean, (0, seg_len - clean.size))
        clean = clean[:seg_len]
        clean = clean / (np.max(np.abs(clean)) + 1e-9) * 0.7

        wet = apply_rir(clean, make_rir(cell['t60'], rng=rng)) if cell['t60'] > 0 else clean
        if cell['real_noise']:
            noise_wav = real_noise_clip(rng, seg_len)
            noise_label = 'dns_real'
        else:
            noise_wav = make_noise(cell['noise'], seg_len, rng)
            noise_label = cell['noise']
        noisy, _ = mix_at_snr(wet, noise_wav, cell['snr'], rng=rng)

        stem = f"{sidx:05d}_{noise_label}_snr{cell['snr']}_t{cell['t60']}"
        write_audio(f'{TESTSET_DIR}/audio/{stem}_noisy.wav', noisy)
        write_audio(f'{TESTSET_DIR}/audio/{stem}_clean.wav', wet)
        records.append({'id': stem, 'noisy': f'audio/{stem}_noisy.wav',
                        'clean': f'audio/{stem}_clean.wav', 'duration_s': round(seg_len/16000, 2),
                        'text': read_transcript(sp), 'source': str(sp),
                        'noise_source': 'real_dns' if cell['real_noise'] else 'synthetic',
                        'snr': cell['snr'], 'noise': noise_label, 't60': cell['t60']})
        sidx += 1

print(f'跳过的超长句子（>{MAX_SEG_SEC}s，理论上不该发生）: {skipped_too_long}')
Path(f'{TESTSET_DIR}/index.json').write_text(
    json.dumps({'sample_rate': 16000, 'min_seg_sec': MIN_SEG_SEC, 'max_seg_sec': MAX_SEG_SEC,
                'per_cell': PER_CELL, 'quick_test': QUICK_TEST, 'version': 'v2_dns_real_noise',
                'records': records},
               ensure_ascii=False, indent=1), encoding='utf-8')
n_real = sum(1 for r in records if r['noise_source'] == 'real_dns')
print(f'测试集 {len(records)} 个样本（真实噪声 {n_real} / 合成噪声 {len(records)-n_real}） → {TESTSET_DIR}')
!du -sh "{TESTSET_DIR}"

### 校验：确认参考文本没有被截断

跟 v1 一样，这一步不是可选的。用 VAD 检测每条**干净参考音频**末尾 0.5 秒是否仍处于
说话状态，为 0 才能放心用这批数据算 CER（详见 `docs/ISSUES.md` I-21）。

In [ ]:
vad = build_vad('energy')
tail_speaking = 0
for r in tqdm(records[:120], desc='抽查末尾截断'):
    clean = read_audio(f'{TESTSET_DIR}/{r["clean"]}')
    flags = vad.process_signal(clean)
    if flags[-31:].mean() > 0.5:
        tail_speaking += 1

print(f'抽查 120 条，末尾仍在说话（疑似截断）: {tail_speaking} 条')
assert tail_speaking == 0, (
    '还有样本在被截断！说明 MAX_SEG_SEC 不够大，或者某条语音本身异常长，'
    '需要回头调大 MAX_SEG_SEC 重新生成这批数据，不要带着这个问题继续往下走。'
)
print('校验通过：抽查样本里没有发现文本-音频错位。')

## 5. 打包测试集，下载到本地

In [ ]:
!cd "{DRIVE}" && rm -f testset.zip && zip -q -r testset.zip testset && ls -lh testset.zip
print()
print('下一步：')
print('  1. 继续跑 02_train.ipynb（数据清单已就绪）')
print(f'  2. 有空时从 Drive 下载 {DRIVE}/testset.zip，解压到本地项目的 data/testset_v2/ 下')
print('     （注意跟 v1 的 data/testset/ 分开放，本地对比脚本需要同时读两份）')